# QLoRA Fine-Tuning — Command-R7B Arabic → SEO Meta Descriptions

Fine-tunes [`CohereLabs/c4ai-command-r7b-arabic-02-2025`](https://huggingface.co/CohereLabs/c4ai-command-r7b-arabic-02-2025) to generate Arabic SEO meta descriptions from article text, using **QLoRA** (PEFT adapters on a 4-bit NF4 double-quantized base) with TRL's `SFTTrainer`.

| | |
|--|--|
| **Training data** | [`ibrahim-nada/mawdoo3-seo-preprocessed-cleaned`](https://huggingface.co/datasets/ibrahim-nada/mawdoo3-seo-preprocessed-cleaned) — prompt/completion pairs from a scraped Arabic article corpus |
| **Published adapter** | [`ibrahim-nada/cmdr7b-ar-seo-qlora-v1-2025-12-20_19.08.11`](https://huggingface.co/ibrahim-nada/cmdr7b-ar-seo-qlora-v1-2025-12-20_19.08.11) |
| **Tracking** | Weights & Biases, checkpoints pushed to HF Hub every 100 steps |
| **Environment** | Google Colab GPU runtime |

The resulting adapter is served by the `seo_generation` module of [nlp-semantic-lab](https://github.com/IbrahimMNada/nlp-semantic-lab).

## 1. Environment Setup

GPU check and dependencies — Transformers, PEFT, TRL, bitsandbytes, W&B.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -U \
  torch \
  transformers \
  datasets \
  peft \
  trl \
  accelerate \
  bitsandbytes \
  sentencepiece \
  wandb

In [ ]:
from google.colab import userdata
from datasets import load_dataset
from huggingface_hub import login
import matplotlib.pyplot as plt
import numpy as np
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datetime import datetime

## 2. Load Dataset

Prompt/completion pairs are loaded from the Hugging Face Hub. The dataset was prepared upstream (scraping, Arabic normalization, cleaning) in the main project.

In [ ]:
# Load dataset from Hugging Face Hub
repo_id = "ibrahim-nada/mawdoo3-seo-preprocessed-cleaned"  # Replace with your actual repo name
split = 'train'  # Options: 'train', 'validation'

# Get token from environment variable
hf_token = userdata.get("HF_TOKEN")

print(f"Loading dataset from Hugging Face Hub: {repo_id}")
print(f"Split: {split}")

if hf_token:
    dataset = load_dataset(repo_id, split=split, token=hf_token)
else:
    dataset = load_dataset(repo_id, split=split)

print(f"\n✓ Dataset loaded: {len(dataset)} examples")
print(f"Features: {list(dataset.features.keys())}")

## 3. Load Tokenizer

Right-side padding, with `eos_token` used as the pad token (the base model ships without one).

In [ ]:
# Load the tokenizer
login(token=userdata.get("HF_TOKEN"))
tokenizer = AutoTokenizer.from_pretrained(
    "CohereLabs/c4ai-command-r7b-arabic-02-2025",
    use_fast=True,
    token=userdata.get("HF_TOKEN")
)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Tokenizer loaded: {tokenizer.__class__.__name__}")

## 4. Data Filtering & Token Statistics

Keep articles that fit comfortably in the 1024-token context window:
- **≤ 450 words** (fast pre-filter before tokenizing)
- **> 10 tokens** (drop degenerate/near-empty samples)

Token-count percentiles are printed to confirm the sequence-length budget.

In [ ]:
def count_words_batch(batch):
    return {
        "word_count": [
            len(re.findall(r'\S+', text))
            for text in batch["clean_prompt"]
        ]
    }

dataset = dataset.map(
    count_words_batch,
    batched=True,
    batch_size=512,
    desc="Counting words"
)

In [ ]:
starter_dataset = dataset.filter(
    lambda x: x["word_count"] <= 450,
    desc="Filtering clean_prompt <= 450 words"
)

In [ ]:
len(starter_dataset)

In [ ]:
def count_tokens_batch(batch):
    enc = tokenizer(
        batch['clean_prompt'],
        add_special_tokens=False,
        truncation=False,
        padding=False,
    )
    return {
        "token_count": [len(ids) for ids in enc["input_ids"]]
    }


starter_dataset = starter_dataset.map(
    count_tokens_batch,
    batched=True,
    batch_size=512,
    desc="Counting tokens"
)

In [ ]:
starter_dataset = starter_dataset.filter(lambda x: x["token_count"] > 10)

token_counts = np.array(starter_dataset["token_count"])

stats = {
    "count": len(token_counts),
    "min": int(token_counts.min()),
    "max": int(token_counts.max()),
    "mean": float(token_counts.mean()),
    "median": float(np.median(token_counts)),
    "p90": float(np.percentile(token_counts, 90)),
    "p95": float(np.percentile(token_counts, 95)),
    "p99": float(np.percentile(token_counts, 99)),
}

for k, v in stats.items():
    print(f"{k:>6}: {v}")

In [ ]:
starter_dataset.to_json("starter_dataset.jsonl", orient="records")

## 5. Experiment Tracking (Weights & Biases)

In [ ]:
import wandb

wandb.login()
wandb.init(
    project="cmdr7b-arabic-seo",
    name="cmdr7b-ar-seo-qlora-v1",
    config={
        "base_model": "CohereLabs/c4ai-command-r7b-arabic-02-2025",
        "task": "arabic_seo_generation",
        "max_seq_length": 1024,
        "training_type": "QLoRA + PEFT",
    }
)

## 6. Run Configuration & Hyperparameters

QLoRA setup: `r=16`, `alpha=32`, `dropout=0.1` on **all** attention (`q/k/v/o_proj`) and MLP (`gate/up/down_proj`) projections — 3 epochs, batch size 8, LR `1e-4` with cosine schedule and `paged_adamw_32bit`. `bf16` is used automatically on Ampere+ GPUs, `fp16` otherwise.

In [ ]:
MODEL_ID = "CohereLabs/c4ai-command-r7b-arabic-02-2025"

In [ ]:
PROJECT_NAME = "cmdr7b-ar-seo-qlora-v1"
HF_USER = "ibrahim-nada"

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyper-parameters - overall

EPOCHS = 3
BATCH_SIZE = 8
MAX_SEQUENCE_LENGTH = 1024
GRADIENT_ACCUMULATION_STEPS = 1

# Hyper-parameters - QLoRA

LORA_R = 16
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

# Hyper-parameters - training

LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

# bf16 on Ampere+ (compute capability >= 8), fp16 otherwise
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# Tracking

LOG_STEPS = 5
SAVE_STEPS = 100
LOG_TO_WANDB = True

## 7. Load Base Model (4-bit NF4)

The 7B base is loaded with double-quantized NF4 so it fits on a single Colab GPU. `use_cache` is disabled for gradient checkpointing compatibility during training.

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
)

model.config.use_cache = False

## 8. Training & LoRA Configuration

`SFTConfig` handles checkpointing (every 100 steps, pushed to the Hub with `hub_strategy="every_save"`) and W&B reporting. The LoRA adapter config targets all attention and MLP projections.

In [ ]:
from trl import SFTTrainer, SFTConfig

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
)

In [ ]:
from peft import LoraConfig
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

## 9. Prompt Formatting

Each sample is rendered into a structured template with explicit markers, so inference can prompt with the same prefix and stop at `<SEO_OUTPUT_PREFIX>`:

```
<SEO_PROMPT_PREFIX>
اكتب وصف ميتا للمقال التالي

<ARTICLE_TEXT>
{article}

<SEO_OUTPUT_PREFIX>
{meta_description}
```

In [ ]:
def format_prompt_completion(batch):
    texts = []

    for article, completion in zip(batch["clean_prompt"], batch["completion"]):
        article = (article or "").strip()
        completion = (completion or "").strip()

        if not article or not completion:
            texts.append(None)
            continue

        text = (
            "<SEO_PROMPT_PREFIX>\n"
            "اكتب وصف ميتا للمقال التالي\n\n"
            "<ARTICLE_TEXT>\n"
            f"{article}\n\n"
            "<SEO_OUTPUT_PREFIX>\n"
            f"{completion}"
        )

        texts.append(text)

    return {"text": texts}


In [ ]:
train_dataset = starter_dataset.map(
    format_prompt_completion,
    batched=True,
    batch_size=512,
    remove_columns=['prompt',
 'completion',
 'index',
 'clean_prompt',
 'word_count',
 'token_count'],
    desc="Formatting prompts"
)

In [ ]:
train_dataset.column_names

In [ ]:
# Sanity-check a formatted sample
train_dataset[44]

## 10. Train & Push to Hub

Runs the fine-tune and pushes the final adapter to the Hugging Face Hub (checkpoints were already synced every 100 steps during training).

In [ ]:
fine_tuning = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_parameters,
    args=train_parameters
)

In [ ]:
# Fine-tune!
fine_tuning.train()

# Push the final adapter to Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

In [ ]:
if LOG_TO_WANDB:
    wandb.finish()